# Basic MjSpec Examples
- Use mjSpec to generate MuJoCo model

In [1]:

import os
import sys
import numpy as np
import time
import mujoco
sys.path.append(os.path.abspath('../'))
# from pp_base_mujoco.VIEWER import MUJOCOGLVIEWER
from pp_base_mujoco.VIEWER import *

import xml.etree.ElementTree as ET
from lxml import etree

In [2]:
def print_xml(xml_input,color=True):
    if isinstance(xml_input, ET.Element):
        rough_string = ET.tostring(xml_input, encoding='unicode')
    else:
        rough_string = xml_input

    parser = etree.XMLParser(remove_blank_text=True)
    tree = etree.fromstring(rough_string, parser=parser)
    pretty_xml = etree.tostring(tree, pretty_print=True, encoding='unicode')
    print(pretty_xml)

#### 1. Declare Spec: empty spec / from string

In [3]:
# simple spec
spec = mujoco.MjSpec()
print(spec) # mj spec object
print_xml(spec.to_xml())

<mujoco model="MuJoCo Model">
  <compiler angle="radian"/>
  <worldbody/>
</mujoco>



In [4]:
path = '../asset/floor_white_gray.xml'
spec = mujoco.MjSpec.from_file(path)

#### 2. Add
- add body: adding empty body
    - add geom: adding geom

In [5]:
# add specific body to worldbody
body = spec.worldbody.add_body(
    name="base_link",
    pos = np.zeros((3)),
    euler = [0, 0.8, 0] # auto-fixed to quat
)
print_xml(spec.to_xml())

print(body) # returns spec body object
print(body == spec.body("base_link"))

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
  </asset>
  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" group="1" material="groundplane"/>
    <light pos="0 0 5.5" dir="0 0 -1" type="directional" castshadow="false" diffuse="0.5 0.5 0.5"/>
    <body name="base_link" quat="0.999976 0 0.00698126 0"/>
  </worldbody>
</mujoco>

True


In [6]:
# add geom to body
geom = body.add_geom(
    name="base_link_geom",
    pos = [0.0, 0.0, 0.1],
    type=mujoco.mjtGeom.mjGEOM_BOX,
    size = [0.1, 0.1, 0.1],
    rgba=[0,1,0,1]
)
spec.body("base_link").add_site(
    name="base_link_site"
)

print(geom)
print(body)
print_xml(spec.to_xml())

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
  </asset>
  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" group="1" material="groundplane"/>
    <light pos="0 0 5.5" dir="0 0 -1" type="directional" castshadow="false" diffuse="0.5 0.5 0.5"/>
    <body name="base_link" quat="0.999976 0 0.00698126 0">
      <geom name="base_link_geom" size="0.1 0.1 0.1" pos="0 0 0.1" type="box" rgba="0 1 0 1"/>
      <site name="base_link_site" pos="0 0 0"/>

In [7]:
# add hierarchical body -> under body 1
body2 = body.add_body(
    name="link1",
    pos = [0.0,0.0,0.3],
)

body2.add_geom(
    name="link1_geom",
    pos = [0.0,0.0,0.1],
    type=mujoco.mjtGeom.mjGEOM_SPHERE,
    size = [0.1, 0.0, 0.0],
    rgba=[1,0,0,1]
)

print_xml(spec.to_xml())

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
  </asset>
  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" group="1" material="groundplane"/>
    <light pos="0 0 5.5" dir="0 0 -1" type="directional" castshadow="false" diffuse="0.5 0.5 0.5"/>
    <body name="base_link" quat="0.999976 0 0.00698126 0">
      <geom name="base_link_geom" size="0.1 0.1 0.1" pos="0 0 0.1" type="box" rgba="0 1 0 1"/>
      <site name="base_link_site" pos="0 0 0"/>

#### 3. Frame & Attach

In [8]:
frame1 = spec.worldbody.add_frame(pos=[0,3,3], quat=[0, 0, 0, 1])

arena_xml = """
<mujoco>
<worldbody>
    <body name="box" pos="0 0 0">
        <geom type="box" size="1 1 1"/>
    </body>
</worldbody>
</mujoco>
"""

additional_spec = mujoco.MjSpec.from_string(arena_xml)
body3 = additional_spec.body('box')

body4 = frame1.attach_body(body3, 'attached-', '-1') # body object, prefix, postfix

print_xml(spec.to_xml())

<mujoco model="floor and sky">
  <compiler angle="radian"/>
  <visual>
    <headlight diffuse="0.6 0.6 0.6" specular="0.9 0.9 0.9"/>
    <rgba haze="0.15 0.25 0.35 1"/>
  </visual>
  <default>
    <default class="attached-main-1"/>
  </default>
  <asset>
    <texture type="skybox" colorspace="auto" builtin="gradient" rgb1="0.4 0.5 0.9" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" colorspace="auto" name="groundplane" builtin="checker" mark="edge" rgb1="0.9 0.9 0.9" rgb2="0.8 0.8 0.8" markrgb="0.8 0.8 0.8" width="1000" height="1000"/>
    <material name="groundplane" texture="groundplane" texuniform="true" reflectance="0.1"/>
  </asset>
  <worldbody>
    <geom name="floor" size="0 0 0.05" type="plane" group="1" material="groundplane"/>
    <light pos="0 0 5.5" dir="0 0 -1" type="directional" castshadow="false" diffuse="0.5 0.5 0.5"/>
    <body name="base_link" quat="0.999976 0 0.00698126 0">
      <geom name="base_link_geom" size="0.1 0.1 0.1" pos="0 0 0.1" type="box" 

In [9]:
model = spec.compile()
data = mujoco.MjData(model)

In [10]:
""" MAIN LOOP """

# create python viewer object
viewer = MUJOCOGLVIEWER(model, data)

# data reset
mujoco.mj_resetData(model, data)

while viewer.is_alive():
        mujoco.mj_step(model, data)
        viewer.render()

# close
viewer.close()